# DR5 Prediction

## Dataset

In [ ]:
from utils import make_train_val_datasets,format_time
import time
import numpy as np
import torch

data_folder = r".\data\P\train"
device = 'cuda' if torch.cuda.is_available() else 'cpu'

start_time = time.time()
train_ds, val_ds, global_stats = make_train_val_datasets(
    data_folder, 
    num_workers=1, 
    step_size=10, 
    window_size=30
)
print(f"Dataset Time: {format_time(time.time() - start_time)}")

# Do not uncomment this during review, as it will overwrite the file.
# np.savez("global_stats_P.npz", **global_stats)

lumped_size = train_ds[0]['lumped'].shape[1]
point_size = train_ds[0]['point'].shape[0]
print('lumped_size: ', lumped_size)
print('point_size: ', point_size)


## Training

In [ ]:
from utils import BatteryMFT, train_battery_model
import time

model = BatteryMFT(
    lumped_size=lumped_size, 
    point_size=point_size, 
    hidden_size=256, 
    encoder_method='lstm'   # best_method
).to(device)

start_train = time.time()

model = train_battery_model(
    model=model,
    save_dir=r"./model_demo",
    train_dataset=train_ds,
    val_dataset=val_ds,
    mode='dr5_prediction',
    batch_size=64, 
    epochs=100,
    lr=1e-3, 
    device=device,
    patience=20,
    lambda_recon=0.1,
    lambda_soc=0.5
)

train_time = time.time() - start_train
print(f"Training Time: {format_time(train_time)}")


# Transfer Learning

## PFS

In [ ]:
from utils import BatteryMFT, format_time, make_train_val_datasets, train_battery_model_TL
import time
import numpy as np
import torch

stats = np.load("global_stats_P.npz", allow_pickle=True)
data_folder = r"data\R\train"
device = 'cuda' if torch.cuda.is_available() else 'cpu'

start_time = time.time()
train_ds_tl, val_ds_tl, global_stats = make_train_val_datasets(
    folder_dir=data_folder, 
    global_stats=stats, 
    step_size=10,
    num_workers=1
)
data_time = time.time() - start_time
print(f"Dataset Time: {format_time(data_time)}")

print("="*8 + " Transfer: PFS " + "="*8)
model = BatteryMFT(
    lumped_size=8,
    point_size=21,
    hidden_size=256,
    encoder_method='lstm'
)
checkpoint = torch.load(r".\model\best_dr5_prediction.pth")
model.load_state_dict(checkpoint)

print("❄️ Freezing feature extraction layers...")
for name, param in model.named_parameters():
    if any(key in name for key in ["lumped_encoder", "recon_head"]):
        param.requires_grad = False
    else:
        param.requires_grad = True
        print(f"🔥 Active: {name}")

start_infer = time.time()
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), 
                              lr=5e-4, weight_decay=1e-4)

model_final = train_battery_model_TL(
    model, 
    save_dir=r"./model_TL_demo/R_PFS", 
    train_dataset=train_ds_tl, 
    val_dataset=val_ds_tl, 
    optimizer=optimizer,
    batch_size=64, 
    epochs=100, 
    patience=15, 
    device=device,
)

infer_time = time.time() - start_infer
print(f"Training Time: {format_time(infer_time)}")

## LLR

In [ ]:
from utils import BatteryMFT, format_time, make_train_val_datasets, train_battery_model_TL
import time
import numpy as np
import torch

stats = np.load("global_stats_P.npz", allow_pickle=True)
data_folder = r"data\R\train"
device = 'cuda' if torch.cuda.is_available() else 'cpu'

start_time = time.time()
train_ds_tl, val_ds_tl, global_stats = make_train_val_datasets(
    folder_dir=data_folder, 
    global_stats=stats, 
    step_size=10,
    num_workers=1
)
data_time = time.time() - start_time
print(f"Dataset Time: {format_time(data_time)}")

print("="*8 + " Transfer: LLR " + "="*8)
model = BatteryMFT(
    lumped_size=8,
    point_size=21,
    hidden_size=256,
    encoder_method='lstm'
)
checkpoint = torch.load(r".\model\best_dr5_prediction.pth")
model.load_state_dict(checkpoint)

start_infer = time.time()
optimizer = torch.optim.Adam([
    {'params': model.lumped_encoder.parameters(), 'lr': 1e-5},
    {'params': model.soc_rnn.parameters(), 'lr': 1e-4},
    {'params': model.dr5_head.parameters(), 'lr': 1e-3},
])

model_final = train_battery_model_TL(
    model, 
    save_dir=r"./model_TL_demo/R_LLR", 
    train_dataset=train_ds_tl, 
    val_dataset=val_ds_tl, 
    optimizer=optimizer,
    batch_size=64, 
    epochs=100, 
    patience=15, 
    device=device,
)

infer_time = time.time() - start_infer
print(f"Training Time: {format_time(infer_time)}")

# Test

## E2E Model (Fig.4)

In [ ]:
from utils import BatteryMFT, evaluate_multitask, make_single_test_dataset
import numpy as np
import torch

file_path = r".\data\P\test\discharge_segment_P218\dis_seg_113_1.pkl"
model_path = r".\model\best_dr5_prediction.pth"
save_dir = r"./results"

# ==============================================
stats = np.load("global_stats_P.npz", allow_pickle=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_path = torch.load(model_path)

test_dataset_seg = make_single_test_dataset(
    file_path=file_path, 
    global_stats=stats,
    step_size=1,
)

model = BatteryMFT(
    lumped_size=8,
    point_size=21,
    hidden_size=256,
    encoder_method='lstm'
)
model.load_state_dict(model_path)
model.to(device)

evaluate_multitask(
    model, test_dataset_seg, device, stats,
    save_dir=save_dir
    )


## TL (Fig.5)

In [ ]:
from utils import BatteryMFT, evaluate_multitask, make_single_test_dataset
import numpy as np
import torch

TL_lst = ['PFS','LLR']

file_path = r".\data\R\test\discharge_segment_R005\dis_seg_1021_5.pkl"
stats = np.load("global_stats_P.npz", allow_pickle=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
test_dataset_seg = make_single_test_dataset(
    file_path=file_path, 
    global_stats=stats,
    step_size=1,
)

for strategy in TL_lst:
    model_path = rf"./model/R_{strategy}\best_TL.pth"
    save_dir = rf"./results_TL/R_{strategy}"
    model_path = torch.load(model_path)

    model = BatteryMFT(
        lumped_size=8,
        point_size=21,
        hidden_size=256,
        encoder_method='lstm'
    )
    model.load_state_dict(model_path)
    model.to(device)

    evaluate_multitask(
        model, test_dataset_seg, device, stats,
        save_dir=save_dir
        )
